# Методы статьи на CAMS CO

Исходные архитектуры из commit 58f40c4 (включая финальную интерполяцию), loss MSE + 0.2 MAE. Это новый эксперимент на CAMS: временное разбиение 2020–2021 / 2022 / 2023, 100 эпох без early stopping. Это не воспроизведение прежних чисел на WRF-Chem.
Все 10 методов статьи; размерности 8,16,32,64. TT-SVD считается отдельно по пяти парам рангов. PCA запускается первой. UMAP использует существующий KNN decoder, обученный только на train; это явно записано в протоколе. Один seed — пилотный запуск.

Все параметры редактируются в CONFIG следующей ячейки. Значения обучения по умолчанию взяты из configs/config.yaml; изменения YAML автоматически сюда не подхватываются. Метрики качества — только validation и test, отдельного train scoring нет.


In [ ]:
# При необходимости выполните один раз и перезапустите kernel:
# %pip install numpy pandas matplotlib netCDF4 scipy scikit-learn torch PyYAML PyWavelets umap-learn tqdm psutil
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/models.py').exists()), None)
assert ROOT is not None, 'Откройте ноутбук внутри репозитория DeepCompreesion'
sys.path.insert(0,str(ROOT))
from scripts.cams_experiment import unpack, inspect_files, prepare, split_days
from scripts.cams_article_experiment import ARTICLE_METHODS, experiment_directory, run_article_experiments
from src.models import get_model

CONFIG = {
    'data_dir': 'data/cams_co_cross_year',
    'variable': None,  # автоопределение co / co_conc; при необходимости задайте имя
    'selected_levels': [0, 250, 500, 1000, 2000, 5000], # эксперимент на шести высотах
    'center_lat_lon': [51.0, 10.0],  # географический центр домена CAMS Europe
    'crop_shape': [96, 84],         # реальные ячейки, без интерполяции
    'train_day_counts': [], # один запуск на всём train pool (2020–2021)
    'split_years': {'train': [2020, 2021], 'validation': 2022, 'test': 2023},
    'evaluation_days_per_month': 7, # из каждого скачанного месяца val/test
    'gap_days': 1,
    'subset_seed': 42,
    'latent_dims': [8, 16, 32, 64],
    'methods': list(ARTICLE_METHODS),
    'evaluation_splits': ['validation', 'test'], # train-метрики не считаются
    'seeds': [42],                 # пилот; для оценки разброса задайте [42, 43, 44] ДО запуска
    'epochs': 100, # минимум validation MSE + beta*MAE; без early stopping
    'batch_size': 32,
    'learning_rate': 0.001,
    'log_every': 1,
    'weight_decay': 0.000001,
    'dropout': 0.1,
    'mae_weight': 0.2, # без весов по высоте
    'tt_ranks': [[2,4], [4,8], [8,8], [8,16], [16,16]],
    'wavelet': 'haar',
    'umap_inverse_method': 'knn',
    'umap_inverse_neighbors': 5,
    'umap_evaluation_batch_size': 64,
    'device': 'auto', # auto / cuda / cpu (baseline-методы используют CPU)
    'cpu_threads': 4,
    'benchmark_frames': 64, # одинаковые кадры val/test для всех методов
    'benchmark_batch_size': 8,
    'benchmark_warmup': 1,
    'benchmark_repeats': 3,
    'memory_sample_interval': 0.02, # секунды между измерениями RSS
    'grad_clip_norm': None, # исходный протокол; для защиты можно задать 1.0, это меняет обучение
}
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))




## 1. Распаковать и посмотреть содержимое
Архивы извлекаются в `data/cams_co/unpacked/`. Повторный запуск пропускает завершённую распаковку. Число кадров в таблице относится к конкретному файлу: если высоты лежат в разных файлах, складывать эти числа нельзя. Следующая ячейка собирает уникальные timestamps и высоты и выявляет дубли/пропуски.


In [ ]:
files = unpack(ROOT/CONFIG['data_dir'], CONFIG['selected_levels'])
inventory, records, reference = inspect_files(files, CONFIG['variable'], CONFIG['selected_levels'])
display(inventory)
print('Исходная сетка (lat, lon):',len(reference[0]),len(reference[1]))
print('Единицы CO:',reference[2],'; единицы высоты:',reference[3])
print('Всего уникальных timestamps:',len({t for r in records for t in r['dates']}))
print('Высоты:',sorted({float(z) for r in records for z in r['levels']}))


## 2. Подготовить регион и показать поля
На диск сохраняются массивы `(time, level, latitude, longitude)` в float32. В память читается один пространственный кадр выбранного региона, не весь месячный файл.

Нормализация совпадает по смыслу с предыдущим экспериментом: min–max для каждого кадра и высоты. Это обратимое преобразование, использующее дополнительные `2 × число высот` чисел на кадр; учитываем их в таблице. Физические метрики вычисляются после обратного преобразования в единицах исходного файла. Метрики CAMS нельзя напрямую смешивать с WRF-Chem.


In [ ]:
selected_files = sorted({record['path'] for record in records})
OUTPUT = experiment_directory(ROOT,CONFIG,selected_files)
inventory.to_csv(OUTPUT/'inventory.csv',index=False)
raw, X, minima, scales, frames, summary = prepare(records,reference,CONFIG,OUTPUT)
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Результаты:',OUTPUT)
fig, axes = plt.subplots(1,3,figsize=(15,4))
levels_to_show = [0,len(summary['levels'])//2,len(summary['levels'])-1]
for ax,z in zip(axes,levels_to_show):
    im=ax.imshow(raw[0,z],origin='upper',aspect='auto')
    ax.set_title(f"Первый кадр, высота {summary['levels'][z]} {summary['level_units']}")
    ax.set_xlabel('Индекс longitude'); ax.set_ylabel('Индекс latitude')
    fig.colorbar(im,ax=ax,label=summary['units'])
fig.tight_layout(); fig.savefig(OUTPUT/'first_frame.png',dpi=150); plt.show()
plt.figure(figsize=(12,3))
plt.plot(frames.time,np.mean(raw,axis=(1,2,3)))
plt.ylabel(f"Среднее CO, {summary['units']}"); plt.xlabel('Время'); plt.tight_layout()
plt.savefig(OUTPUT/'co_time_series.png',dpi=150); plt.show()


## Исходные архитектуры статьи
SAM3D и plain CAE: одинаковые блоки, различие только в наличии attention. Фактическая форма входа определяется загруженными высотами. Слои decoder и финальная интерполяция восстановлены из первоначального кода.


In [ ]:
import torch
for name in ['ArticlePlainAutoencoder', 'ArticleSAMAutoencoder']:
    model = get_model(name, latent_dim=CONFIG['latent_dims'][-1],
                      input_shape=(1, *X.shape[1:]), dropout_rate=CONFIG['dropout'])
    print(name)
    print(model)
    print('Parameters:', sum(p.numel() for p in model.parameters()))
    model.eval()
    with torch.no_grad():
        sample = torch.from_numpy(np.array(X[:1], dtype='float32'))[:, None]
        reconstruction, code = model(sample)
    assert reconstruction.shape == sample.shape
    print('Shapes:', tuple(sample.shape), tuple(code.shape), tuple(reconstruction.shape))
    del model, sample, reconstruction, code



## 3. Зафиксировать разбиение
Полные 2020–2021 годы образуют train pool. Из каждого скачанного месяца 2022 выбираются 7 равномерно расположенных дней для validation, а из каждого скачанного месяца 2023 — 7 дней для test. Test нельзя использовать для выбора архитектуры или эпохи.

Пространственные патчи из одного timestamp не размножаются между выборками. Соседние дни всё равно могут коррелировать; это пилотное временное разбиение, не независимые атмосферные реализации. PCA fit выполняется только на train.


In [ ]:
splits, counts = split_days(frames,CONFIG,OUTPUT)
display(pd.read_csv(OUTPUT/'partitions.csv').groupby('partition').agg(frames=('frame','size'),days=('day','nunique'),first=('time','min'),last=('time','max')))
display(pd.DataFrame([{'train_days':n,'frames':len(splits[f'train_{n}'])} for n in counts]))
assert min(len(splits[f'train_{n}']) for n in counts) >= max(CONFIG['latent_dims']), 'Увеличьте минимальный train или уменьшите latent_dims для PCA'
print('Запусков:', len(CONFIG['seeds']) * sum(len(CONFIG['tt_ranks']) if m == 'TT-SVD' else len(CONFIG['latent_dims']) for m in CONFIG['methods']))


## Обучение и оценка всех методов
AE обучаются с loss MSE + beta*MAE без весов по высоте. Все 100 эпох; checkpoint по минимуму validation loss. Метрики только validation/test. Полный набор может считаться долго, особенно UMAP и TT-SVD. Завершённые запуски переиспользуются в пределах того же протокола.


In [ ]:
metrics = run_article_experiments(X,raw,minima,scales,frames,splits,counts,CONFIG,OUTPUT)
display(metrics[['method','latent_dim','tt_ranks','split','mse','mae','rmse','relative_l2','ssim','physical_rmse','payload_bytes','best_epoch']])


## Итоговые таблицы
TT-ранги не равны latent_dim. Сравнивайте фактические payload_bytes: DCT/Wavelet включают индексы, TT — все ядра, Interpolation — фактическую грубую сетку. Общие веса/базисы и train-база UMAP decoder не входят в покадровый payload. Размер нормировочных параметров указан отдельно.


In [ ]:
for split in ['validation', 'test']:
    table = metrics[metrics.split == split]
    print(split, '— векторные методы')
    display(table[table.method != 'TT-SVD'])
    print(split, '— TT-SVD (по рангам)')
    display(table[table.method == 'TT-SVD'])
    table.to_csv(OUTPUT/f'{split}_article_metrics.csv', index=False)
print('Результаты:', OUTPUT)



## Время и память
training_seconds: суммарное время обучающих проходов AE, без validation. fit_seconds: весь вызов обучения/подготовки, включая validation и checkpoint IO. У методов без обучения training_seconds=0; их подготовка измеряется в fit_seconds. Каждый повтор инференса включает encode и decode, CPU↔GPU копирование, синхронизацию CUDA; загрузка с диска, метрики и CSV исключены. В таблице медианы повторов, отдельно сохранены все повторы. Прогрев исключён.
RAM — выборочный пик RSS всего процесса Jupyter, включая загруженные данные и оставшиеся аллокации; это не изолированная память модели. Также записаны стартовый RSS и прирост. GPU — пики PyTorch allocated/reserved во время фазы; у CPU-методов отсутствуют. Оборудование, версии и число CPU threads сохранены в protocol.json. Не запускайте параллельные вычисления в том же kernel при замерах.


In [ ]:
profile_columns = ['method','latent_dim','tt_ranks','seed','split','device','training_seconds','training_validation_seconds','fit_seconds','encode_ms_per_frame','decode_ms_per_frame','inference_ms_per_frame','inference_frames_per_second','fit_ram_peak_mib','fit_ram_peak_delta_mib','fit_gpu_peak_allocated_mib','inference_ram_peak_mib','inference_gpu_peak_allocated_mib','payload_bytes','scaling_bytes_per_frame']
profile = metrics[profile_columns]
display(profile)
profile.to_csv(OUTPUT/'timing_memory.csv', index=False)
